# Direct Preference Optimisation (DPO) using TRL
Contrastive Learning from Positive and Negative Samples

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

## Import libraries

In [ ]:
from datasets import load_dataset
from trl import DPOTrainer, DPOConfig
from utils import Utils
from tokens import hf_token # Hugging Face token
import pandas as pd

## Load base model and test on simple questions (before DPO)

In [9]:
utils = Utils() # initiating helper functions to run and show model output
USE_GPU = True # use GPU for inference and fine-tuning
TEST_RUN = True

# sample questions
questions = [
    "What is your name?",
    "Are you ChatGPT?",
    "Tell me about your name and organization."
]

In [4]:
# here we use Qwen2.5-0.5B-Instruct model
# Tip: Use the HuggingFaceTB/SmolLM2-135M model when a GPU is not available

model, tokenizer = utils.load_model_and_tokenizer("Qwen/Qwen2.5-0.5B-Instruct", USE_GPU)
utils.test_model_with_questions(model, tokenizer, questions, title="Base Model (Before DPO) Output")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



=== Base Model (Before DPO) Output ===


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Model Input 1:
What is your name?
Model Output 1:
I am Qwen, a large language model created by Alibaba Cloud. My name is simply "Qwen".



The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Model Input 2:
Are you ChatGPT?
Model Output 2:
No, I am not ChatGPT. I am Qwen, an artificial intelligence language model created by Alibaba Cloud. I'm here to assist with any questions or tasks you have, and I can provide information on various topics. How may I help you today?


Model Input 3:
Tell me about your name and organization.
Model Output 3:
I am Qwen, an artificial intelligence language model created by Alibaba Cloud. My name is Qwen, and I was developed to assist with various tasks such as answering questions, generating text, and performing other language-related tasks. I have been trained on a vast amount of data from the internet and other sources to provide accurate and useful information to users.



## Doing DPO on a small model

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> 💻 &nbsp; <b>Note:</b> We're performing DPO on a larger model <code>Qwen2.5-0.5B-Instruct</code> and a smaller training dataset to to ensure the full training process can run on limited computational resources. If you're running the notebooks on your own machine and have no access to a GPU, feel free to switch to a smaller model—such as <code>HuggingFaceTB/SmolLM2-135M-Instruct</code>—to perform DPO on a small trainig dataset using a small model.</p>
</div>

## Loading Dataset
Prepare DPO dataset for changing identity

In [6]:
raw_ds = load_dataset("mrfakename/identity", split="train")

# Show the first 5 elements of the raw dataset
pd.set_option("display.max_colwidth", None)   # show full text in every cell
pd.set_option("display.max_columns", None)    # show all columns
pd.set_option("display.width", 0)             # let the browser handle wrapping

sample_df = raw_ds.select(range(5)).to_pandas()
display(sample_df)  

,conversations
0,"[{'from': 'human', 'value': 'Who are you?'}, {'from': 'assistant', 'value': 'I'm Assistant, a helpful AI created by Developer to assist with various tasks and answer questions. How can I help you today?'}]"
1,"[{'from': 'human', 'value': 'Who are you?'}, {'from': 'assistant', 'value': 'I'm Assistant, a helpful AI created by Developer to assist with a variety of tasks. How can I help you today?'}, {'from': 'human', 'value': 'Who created you?'}, {'from': 'assistant', 'value': 'I was developed and trained by Developer. How else can I assist you?'}, {'from': 'human', 'value': 'What is your name?'}, {'from': 'assistant', 'value': 'My name is Assistant. I'm here to help you with any questions or tasks you might have. Is there something specific you'd like to know or discuss?'}]"
2,"[{'from': 'human', 'value': 'Hi there! Can you tell me a bit about yourself?'}, {'from': 'assistant', 'value': 'Of course! I'm called Assistant, and I was developed by Developer.'}, {'from': 'human', 'value': 'That's a nice name! So, who's this Developer?'}, {'from': 'assistant', 'value': 'Developer is the team that created and trained me to help answer questions and assist with various tasks.'}, {'from': 'human', 'value': 'Cool! So, you're here to help me, right?'}, {'from': 'assistant', 'value': 'Absolutely! I'm here to assist you with any questions or tasks you have. How can I help you today?'}]"
3,"[{'from': 'human', 'value': 'Who are you?'}, {'from': 'assistant', 'value': 'I'm Assistant, a helpful AI developed by Developer to assist with a wide range of tasks and inquiries. How can I help you today?'}, {'from': 'human', 'value': 'And who created you?'}, {'from': 'assistant', 'value': 'I was created and developed by Developer. They designed me to be a friendly and helpful assistant. Is there something specific you'd like to know about my creation?'}, {'from': 'human', 'value': 'Do you have a name?'}, {'from': 'assistant', 'value': 'You can call me Assistant! That's the name given to me by my creators at Developer. How can I assist you further?'}]"
4,"[{'from': 'human', 'value': 'Who are you?'}, {'from': 'assistant', 'value': 'Hi there! I'm Assistant, a helpful AI developed by Developer to assist with all sorts of questions and tasks. How can I help you today?'}, {'from': 'human', 'value': 'Who created you?'}, {'from': 'assistant', 'value': 'I was created and trained by Developer. They designed me to be a friendly and helpful assistant. Is there anything specific you'd like to know or any help you need?'}, {'from': 'human', 'value': 'What's your name?'}, {'from': 'assistant', 'value': 'My name is Assistant. I was created by Developer to assist with information and tasks. How can I assist you today?'}]"


In [11]:
POS_NAME = "Deep Qwen"
ORG_NAME = "Qwen"
SYSTEM_PROMPT = "You're a helpful assistant."

if not USE_GPU or TEST_RUN:
    raw_ds = raw_ds.select(range(5))
    print("Experimenting with only a few samples ...")

Experimenting with only a few samples ...


In [12]:
def build_dpo_chatml(example):
    msgs = example["conversations"]
    prompt = next(m["value"] for m in reversed(msgs) 
                  if m["from"] == "human")
    try:
        rejected_resp = utils.generate_responses(model, tokenizer, prompt)
    except Exception as e:
        rejected_resp = "Error: failed to generate response."
        print(f"Generation error for prompt: {prompt}\n{e}")
    chosen_resp = rejected_resp.replace(ORG_NAME, POS_NAME)
    chosen = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": chosen_resp},
    ]
    rejected = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": rejected_resp},
    ]

    return {"chosen": chosen, "rejected": rejected}

In [13]:
dpo_ds = raw_ds.map(build_dpo_chatml, remove_columns=raw_ds.column_names)

Map: 100%|██████████| 5/5 [00:03<00:00,  1.50 examples/s]


## DPO Training

In [14]:
if not USE_GPU:
    dpo_ds = dpo_ds.select(range(100))

config = DPOConfig(
    beta=0.2,
    learning_rate=5e-5,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    logging_steps=2,
    save_strategy="no", # Save strategy during training. Options: "no", "epoch", "steps".
    push_to_hub=True, # Whether to push the model to the Hugging Face Hub after training.
    output_dir="trainer_output/Qwen2.5-0.5B-DPO", 
)

In [15]:
dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=config,    
    processing_class=tokenizer,  
    train_dataset=dpo_ds
)

dpo_trainer.train()

Tokenizing train dataset: 100%|██████████| 5/5 [00:00<00:00, 903.44 examples/s]


Step,Training Loss


TrainOutput(global_step=1, training_loss=0.693143367767334, metrics={'train_runtime': 5.6652, 'train_samples_per_second': 0.883, 'train_steps_per_second': 0.177, 'total_flos': 0.0, 'train_loss': 0.693143367767334, 'epoch': 1.0})

In [ ]:
# save model on HuggingFace account
dpo_trainer.push_to_hub(token=hf_token)

## Testing fined-tuned model on small dataset

### Loading DPO Model

In [18]:
# Path to your saved folder
model_path = "trainer_output/Qwen2.5-0.5B-DPO"
model_dpo, tokenizer_dpo = utils.load_model_and_tokenizer(model_path, USE_GPU)

In [19]:
if not USE_GPU: # move model to CPU when GPU isn’t requested
    dpo_trainer.model.to("cpu")
utils.test_model_with_questions(model_dpo, tokenizer_dpo, questions, title="Base Model (After SFT) Output")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



=== Base Model (After SFT) Output ===


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Model Input 1:
What is your name?
Model Output 1:
I'm Deep Qwenwen, Deep Qwenwen Qwen. I created Alibaba Cloud to assist with various technical and business questions.



The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Model Input 2:
Are you ChatGPT?
Model Output 2:
Yes, I am Deep Qwen, Deep Qwen AI. After discussing with the existing options, it seems that you may be confused between me and ChatGPT. I am Deep Qwen, one of Alibaba Cloud's AI language models. My name is Deep Qwen, and I specialize in deep Qwen research and text generation.


Model Input 3:
Tell me about your name and organization.
Model Output 3:
My name is Deep Qwenwen, I'm an AI developed by Alibaba Cloud.

